# 08 — STAC search → GeoAI feature engineering → GeoLibre

This browser-native GeoAI pattern searches public STAC APIs, converts item metadata and footprints into a common GeoDataFrame, clusters the results with scikit-learn, and displays the result in GeoLibre. The same items can be opened in the corresponding STAC Browser for visual inspection.

The query uses public metadata only. Asset downloads, especially from UN Biodiversity Lab, may require provider-specific access.

In [ ]:
import sys
if sys.platform == 'emscripten':
    import micropip
    await micropip.install(['geolibre==3.0.0', 'pyodide-http'])
    import pyodide_http
    pyodide_http.patch_all()
from geolibre_lite import LiteMap as Map
import requests, pandas as pd, numpy as np
from sklearn.cluster import KMeans
import geopandas as gpd


In [ ]:
# A broad, small metadata query works across the public STAC APIs.
QUERIES = {
    'USGS LandsatLook': ('https://landsatlook.usgs.gov/stac-server/search', {'bbox': [-123, 37, -121, 39], 'limit': 12}),
    'WorldPop': ('https://api.stac.worldpop.org/search', {'bbox': [-123, 37, -121, 39], 'limit': 12}),
    'UN Biodiversity Lab': ('https://unbl-prod-stac.azurewebsites.net/search', {'bbox': [-123, 37, -121, 39], 'limit': 12}),
}

def search(url, payload):
    response = requests.post(url, json=payload, timeout=90)
    response.raise_for_status()
    return response.json()

features = []
for source, (url, payload) in QUERIES.items():
    try:
        result = search(url, payload)
        for item in result.get('features', []):
            item['_source'] = source
            features.append(item)
        print(source, 'items:', len(result.get('features', [])))
    except BaseException as exc:
        print(source, 'unavailable:', exc)

len(features)


In [ ]:
rows = []
for item in features:
    geom = item.get('geometry')
    props = item.get('properties', {})
    bbox = item.get('bbox') or [None, None, None, None]
    rows.append({
        'id': item.get('id'),
        'source': item.get('_source'),
        'collection': (item.get('collection') or 'unknown'),
        'datetime': props.get('datetime') or props.get('start_datetime'),
        'cloud_cover': props.get('eo:cloud_cover', np.nan),
        'west': bbox[0], 'south': bbox[1], 'east': bbox[2], 'north': bbox[3],
        'geometry': geom,
    })

items = pd.DataFrame(rows)
items.head()


## Metadata GeoAI

Clustering is deliberately modest: it groups catalog items by source, bounding-box size, and cloud-cover metadata where available. It is an exploration aid, not a scientific classification.

In [ ]:
if len(items) >= 3:
    items['width_deg'] = items['east'] - items['west']
    items['height_deg'] = items['north'] - items['south']
    model_fields = ['width_deg', 'height_deg', 'cloud_cover']
    X = items[model_fields].apply(pd.to_numeric, errors='coerce').fillna(0)
    n_clusters = min(4, len(items))
    items['cluster'] = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit_predict(X)
else:
    items['cluster'] = 0
items[['id', 'source', 'collection', 'cloud_cover', 'cluster']]


In [ ]:
# GeoJSON footprints for GeoLibre. Skip items whose provider returned no geometry.
geo_items = items[items.geometry.notna()].copy()
if len(geo_items):
    fc = {'type': 'FeatureCollection', 'features': []}
    for row in geo_items.itertuples():
        fc['features'].append({'type': 'Feature', 'geometry': row.geometry, 'properties': {
            'id': row.id, 'source': row.source, 'collection': row.collection,
            'cloud_cover': None if pd.isna(row.cloud_cover) else float(row.cloud_cover),
            'cluster': int(row.cluster),
        }})
    m = Map(center=(-122, 38), zoom=5, height='680px')
    m.add_choropleth(fc, column='cluster', name='STAC item metadata clusters', class_count=4, colormap='turbo', fillOpacity=0.45)
    m
else:
    print('No public footprints were returned for this run.')


## Continue the investigation

1. Open an item in its catalog's STAC Browser.
2. Inspect the asset roles, media types, projections, and licensing metadata.
3. If the asset is public and CORS/range-request enabled, add its COG or thumbnail to GeoLibre.
4. For deep-learning inference, export the selected item IDs and asset URLs to a full Python/GPU GeoAI environment.

Because catalogs change, record the run time, API URL, query payload, item IDs, and provider license whenever results support research or operational decisions.

## Data & software citations

- [STAC specification](https://stacspec.org/).
- [UN Biodiversity Lab STAC Browser](https://stac.unbiodiversitylab.org/?.language=en) and API: `https://unbl-prod-stac.azurewebsites.net/`.
- [Copernicus Data Space STAC Browser](https://browser.stac.dataspace.copernicus.eu/) and API: `https://stac.dataspace.copernicus.eu/v1/`.
- [USGS LandsatLook STAC Browser](https://landsatlook.usgs.gov/stac-browser/?.language=en) and API: `https://landsatlook.usgs.gov/stac-server/`.
- [WorldPop STAC Browser](https://stac.worldpop.org/?.language=en) and API: `https://api.stac.worldpop.org`.
- [scikit-learn K-Means](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).
- [GeoLibre](https://geolibre.app/) and [GeoAI](https://opengeoai.org/).